Project Settings to change:
1. Enable Identity-Aware Proxy
2. Enable OS Login (Not sure about this, currently in my test project it's disabled, but I heard best practice is to keep it enabled)
   1. https://cloud.google.com/compute/docs/oslogin/set-up-oslogin#gcloud
3. Please consider adding the IAP-secured Tunnel User IAM role (iap.tunnelInstances.accessViaIAP) to start using Cloud IAP for TCP forwarding for better performance.

Run the following in Cloud Shell in GCP to create the VM:

Change the following first:
instance name
metadata should be empty first
service account should be your service account

In [ ]:
gcloud compute instances create drg-data-pipeline \
  --project=drg-pipeline \
  --zone=us-central1-a \
  --machine-type=e2-highmem-8 \
  --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
  --metadata=enable-osconfig=TRUE,startup-script=\#\!/bin/bash$'\n'sudo\ apt-get\ update\ -y$'\n'sudo\ apt-get\ upgrade\ -y$'\n'USER=\"resurreccion_cmc_gmail_com\"$'\n'sudo\ -u\ \$USER\ bash\ -c\ \'code\ tunnel\',enable-oslogin=TRUE \
  --maintenance-policy=MIGRATE \
  --provisioning-model=STANDARD \
  --service-account=271591364028-compute@developer.gserviceaccount.com \
  --scopes=https://www.googleapis.com/auth/cloud-platform \
  --tags=http-server,https-server,lb-health-check \
  --create-disk=auto-delete=yes,boot=yes,device-name=drg-data-pipeline,image=projects/ubuntu-os-cloud/global/images/ubuntu-2404-noble-amd64-v20240809,mode=rw,size=200,type=pd-ssd \
  --shielded-secure-boot \
  --shielded-vtpm \
  --shielded-integrity-monitoring \
  --labels=goog-ec-src=vm_add-gcloud \
  --reservation-affinity=any

Configure VS Code code tunnel to work on the VM

In [ ]:
sudo snap install code --classic
code tunnel
# open the link in the log (either in terminal when the link shows up, or in serial port 1 after the startup script runs) 
# and type in the authentication code XXXX-XXXX

Create ssh keys (run in windows terminal)

In [ ]:
# ssh-keygen -t rsa -f "C:\\Users\\resur\\.ssh\\gcp-carlosresu" -C carlosresu # change carlosresu to your username
# # Passphrase = 1Password

Run this on windows terminal to get the HostName and HostKeyAlias

In [ ]:
# gcloud compute ssh drg-data-pipeline --dry-run --tunnel-through-iap

In [ ]:
# # # Output will be something like this:

# C:\Users\resur>gcloud compute ssh drg-data-pipeline --dry-run --tunnel-through-iap

# No zone specified. Using zone [us-central1-a] for instance: [drg-data-pipeline].

# "C:\Users\resur\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\sdk\putty.exe" -t -i "C:\Users\resur\OneDrive - Philippine Institute for Development Studies\DRG\.ssh\google_compute_engine.ppk" -proxycmd ""C:\\Users\\resur\\.pyenv\\pyenv-win\\versions\\3.12.4\\python.exe" "C:\\Users\\resur\\AppData\\Local\\Google\\Cloud SDK\\google-cloud-sdk\\bin\\..\\lib\\gcloud.py" compute start-iap-tunnel "drg-data-pipeline" "%port" --listen-on-stdin --project=drg-pipeline --zone=us-central1-a --verbosity=warning" resurreccion_cmc_gmail_com@compute.723070816956143024

The below is the config file stored at C:\Users\username\\.ssh\config

In [ ]:
# Host test-drg-data-pipeline # CHANGE INSTANCE NAME
#     HostName compute.723070816956143024 # CHANGE THE NUMBER AFTER THE PERIOD
#     User carlosresu # CHANGE THIS TO YOUR USERNAME
#     IdentityFile "C:\\Users\\resur\\.ssh\\gcp-carlosresu" # CHANGE USERNAME, GENERATE YOUR OWN IDENTITY FILE, UPLOAD PUBLIC KEY TO VM
#     CheckHostIP no
#     HashKnownHosts no
#     HostKeyAlias compute.723070816956143024 # CHANGE THE NUMBER AFTER THE PERIOD
#     IdentitiesOnly yes
#     StrictHostKeyChecking no # START WITH THIS DISABLED, THEN ENABLE AFTER FIRST RUN
#     UserKnownHostsFile "C:\\Users\\resur\\.ssh\\google_compute_known_hosts" # CHANGE USERNAME TO YOUR OWN
#     ProxyCommand "C:\\Users\\resur\\.pyenv\\pyenv-win\\versions\\3.12.4\\python.exe" "C:\\Users\\resur\\AppData\\Local\\Google\\Cloud SDK\\google-cloud-sdk\\bin\\..\\lib\\gcloud.py" compute start-iap-tunnel "drg-data-pipeline" "%p" --listen-on-stdin --project=drg-pipeline --zone=us-central1-a --verbosity=warning # CHANGE USERNAME, PYENV PYTHON VERSION, INSTANCE NAME, PROJ NAME, ZONE
#     ProxyUseFdpass no


Save the above as ~/.ssh/config

Connect with StrictHostKeyChecking no first then enable it later

Then ssh into the VM with the below command

In [ ]:
# ssh drg-data-pipeline

VM Setup commands:

In [ ]:
# Update the package list:
sudo apt update

# Install Jupyter:
sudo apt install jupyter jupyter-core jupyter-client build-essential libcurl4-openssl-dev libssl-dev libxml2-dev libsodium-dev python3-full python3-pip

# upgrade packages
sudo apt upgrade

Python venv

In [ ]:
# python -m venv ~/drg-pipeline/data-cleaning/python-venv
# source  ~/drg-pipeline/data-cleaning/python-venv/bin/activate
# pip install numpy pandas streamlit python_dateutil tabulate rpy2
# deactivate

Setup for RStudio Server

In [ ]:
# sudo apt-get install gdebi-core

# wget https://download2.rstudio.org/server/jammy/amd64/rstudio-server-2024.04.2-764-amd64.deb

# sudo gdebi rstudio-server-2024.04.2-764-amd64.deb

In [ ]:
# sudo systemctl start nginx
# sudo systemctl enable nginx


In [ ]:
# sudo nano /etc/nginx/sites-available/pids-vm.resu.works


In [ ]:
# server {
#     listen 80;
#     server_name pids-vm.resu.works;

#     location / {
#         proxy_pass http://127.0.0.1:8787;
#         proxy_set_header Host $host;
#         proxy_set_header X-Real-IP $remote_addr;
#         proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
#         proxy_set_header X-Forwarded-Proto $scheme;
#     }
# }


In [ ]:
# sudo ln -s /etc/nginx/sites-available/pids-vm.resu.works /etc/nginx/sites-enabled/


In [ ]:
# sudo nginx -t


In [ ]:
# sudo systemctl reload nginx


Turn off anti-bot measures on website first

In [ ]:
# sudo certbot --nginx -d pids-vm.resu.works


Turn on anti-bot measures on website again

In [ ]:
# sudo groupadd rstudio
# sudo adduser carlosresu # See 1Password
# sudo usermod -aG rstudio carlosresu


VM R installation

In [ ]:
# Install R
## Update package list
sudo apt update
sudo apt install -y software-properties-common dirmngr
## Add CRAN GPG Key
wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc | sudo tee -a /etc/apt/trusted.gpg.d/cran_ubuntu_key.asc
## Add CRAN repository
sudo add-apt-repository 'deb https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/'
## Update package list again
sudo apt update
## Install R 4.4.1
sudo apt install -y r-base
## Check R Version
R --version

Make system-wide libraries writable by R

Otherwise, we'd need to rely on renv and I haven't gotten that to work

In [ ]:
sudo chmod -R 777 /usr/local/lib/R/site-library
sudo chmod -R 777 /usr/lib/R/site-library
sudo mkdir -p /home/data
sudo chmod -R 777 /home/data
sudo chown -R root:root /home/data
sudo chmod -R 777 /home/data


R package installation (sudo to install to site-wide library)

In [ ]:
# To start R
sudo R

Package installation

In [ ]:
install.packages("languageserver")
install.packages("jsonlite")
install.packages("rlang")
install.packages("yaml")
install.packages("IRkernel")
install.packages("here")
IRkernel::installspec(user = FALSE)
IRkernel::installspec(user = FALSE)


In [ ]:
quit()
quit()


The following commented out code is for running jupyter lab, not VS code

In [ ]:
# R

In [ ]:
# install.packages("languageserver")
# install.packages("jsonlite")
# install.packages("rlang")
# install.packages("yaml")
# install.packages("IRkernel")
# install.packages("here")


In [ ]:
# jupyter notebook --generate-config
# jupyter notebook stop 
# jupyter notebook
# nano /home/resur/.jupyter/jupyter_notebook_config.py


In [ ]:
# c.NotebookApp.kernel_spec_manager_class = 'jupyter_client.kernelspec.KernelSpecManager'
# c.KernelSpecManager.ensure_native_kernel = False
# c.KernelSpecManager.whitelist = set(['python3', 'ir'])


In [ ]:
# jupyter notebook stop 
# jupyter notebook
# export JUPYTER_PATH=/usr/local/share/jupyter/kernels


In [ ]:
# sudo R

In [ ]:
# install.packages('IRkernel')
# IRkernel::installspec(user = FALSE)
# quit()


Check jupyter kernels

In [ ]:
# Verify R jupyter kernel is usable
jupyter kernelspec list


Install gcloud CLI on the VM

In [ ]:
sudo apt-get update

sudo apt-get install apt-transport-https ca-certificates gnupg curl

curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo gpg --dearmor -o /usr/share/keyrings/cloud.google.gpg

echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt cloud-sdk main" | sudo tee -a /etc/apt/sources.list.d/google-cloud-sdk.list

sudo apt-get update && sudo apt-get install google-cloud-cli

gcloud init

# Login with the service account

Configure git on the VM

In [ ]:
# Configure git on the VM

git config --global user.name "Carlos Miguel Resurreccion"
git config --global user.email resurreccion.cmc@gmail.com

Configure Patch

In [ ]:
# https://console.cloud.google.com/compute/patch/dashboard enable patch
# https://console.cloud.google.com/apis/api/osconfig.googleapis.com/overview
# Manually install OS Config agent
# sudo su -c "echo 'deb http://packages.cloud.google.com/apt google-compute-engine-focal-stable main' > /etc/apt/sources.list.d/google-compute-engine.list"

# sudo apt update
# sudo apt -y install google-osconfig-agent

# # Apply OS Config to all instances in a project
# gcloud compute project-info add-metadata --project drg-pipeline --metadata=enable-osconfig=TRUE

# # Apply OS Config to an individual VM
# gcloud compute instances add-metadata drg-data-pipeline --metadata=enable-osconfig=TRUE

# # Apply OS Config when creating an instance
# gcloud compute instances create drg-data-pipeline --metadata=enable-osconfig=TRUE

# Enable full VM Manager features
gcloud compute os-config project-feature-settings update --project drg-pipeline --patch-and-config-feature-set=full

# Verify it's working
gcloud compute os-config project-feature-settings describe --project drg-pipeline

Clone drg-pipeline into your home folder first

In [ ]:
git clone https://github.com/pids-drg/drg-pipeline

Link /home/data to your username's drg-pipeline/data-cleaning/data folder

In [ ]:
sudo ln -s /home/data /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning # change resurreccion_cmc_gmail_com to your username
# sudo ln -s /home/data /home/carlosresu/drg-pipeline/data-cleaning # change carlosresu to your username

Link /home/resurreccion_cmc_gmail_com/grouper to /home/resurreccion_cmc_gmail_com/drg-pipeline/python_grouper

In [ ]:
git clone https://github.com/pids-drg/grouper

In [ ]:
sudo ln -s /home/resurreccion_cmc_gmail_com/grouper /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning # change resurreccion_cmc_gmail_com to your username
# sudo ln -s /home/carlosresu/grouper /home/carlosresu/drg-pipeline/data-cleaning # change resurreccion_cmc_gmail_com to your username

In [ ]:
sudo ln -s /home/resurreccion_cmc_gmail_com/grouper/libraries /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning
sudo ln -s /home/resurreccion_cmc_gmail_com/grouper/misc /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning
sudo ln -s /home/resurreccion_cmc_gmail_com/grouper/scripts /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning
sudo ln -s /home/resurreccion_cmc_gmail_com/grouper/tests /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning

Current Workaround to signin to google cloud code is:

sign in

copy localhost url thing

curl "localhost url"

Startup Script

In [ ]:
#!/bin/bash
sudo apt-get update -y
sudo apt-get upgrade -y
USER="resurreccion_cmc_gmail_com"
sudo -u $USER bash -c 'code tunnel'

See what startup script is doing

In [ ]:
# sudo journalctl -u google-startup-scripts.service

To store git personal access token in RStudio Server: 

do something with git, login with carlosresu and the PAT,

then

In [ ]:
# su carlosresu #change to your username
# git config --global credential.helper store


Run Thai Grouper programmatically

In [ ]:
# sudo dpkg --add-architecture i386
# sudo apt update
# sudo apt upgrade -y
# sudo apt install wine64 wine32 xvfb xautomation
# sudo apt install xfce4 xfce4-goodies -y
# wget https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb
# sudo apt install --assume-yes ./chrome-remote-desktop_current_amd64.deb


In [ ]:
# sudo dpkg --remove-architecture i386
# sudo apt-get purge --auto-remove wine64 wine32 xvfb xautomation xfce4 xfce4-goodies
# sudo apt-get purge --auto-remove chrome-remote-desktop
# sudo apt-get autoclean
# sudo apt-get autoremove
# rm -f ~/chrome-remote-desktop_current_amd64.deb


In [ ]:
# Xvfb :1 -screen 0 1024x768x16 &
# export DISPLAY=:1


In [ ]:
# wine --version
# winecfg

In [ ]:
# #!/bin/bash

# # Start xvfb
# Xvfb :1 -screen 0 1024x768x16 &
# export DISPLAY=:1

# # Run the Windows application
# wine /path/to/your/application.exe &

# sleep 5  # Wait for the application to load

# # Simulate keyboard input or mouse clicks to select the file
# xte 'key Tab' 'key Tab' 'key Return' &

# # You might need to simulate additional keys depending on how the application works.


In [ ]:
# https://cloud.google.com/compute/docs/instances/nested-virtualization/enabling#gcloud_1

In [ ]:
# gcloud compute instances export drg-data-pipeline \
#   --destination=/home/resurreccion_cmc_gmail_com/drg-pipeline/yaml.yaml \
#   --zone=us-central1-a

# # add the ff:
# advancedMachineFeatures:
#   enableNestedVirtualization: true

# gcloud compute instances update-from-file drg-data-pipeline \
#   --source=/home/resurreccion_cmc_gmail_com/drg-pipeline/yaml.yaml \
#   --most-disruptive-allowed-action=RESTART \
#   --zone=us-central1-a